# 05 — Feature Engineering
Demonstrates all 9 hand-crafted features added by `src/features.py`, compares Feature Set A (with duration) vs Feature Set B (without duration), and saves the processed train/test splits to `data/processed/`.

In [1]:
import sys, warnings
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DIR, TARGET_COL, ENGINEERED_FEATURE_NAMES, PROCESSED_DIR
from src.data_loader import load_dataset
from src.features import encode_target, add_features, get_feature_lists
from src.preprocessing import split_data
from src.utils import ensure_dir

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

df_raw = load_dataset(RAW_DIR / "bank-additional-full.csv")
df_raw = encode_target(df_raw, TARGET_COL)
print(f"Raw shape: {df_raw.shape}")

2026-05-16 21:15:55 | INFO     | src.data_loader | Loaded bank-additional-full.csv — shape (41188, 21)
2026-05-16 21:15:55 | INFO     | src.features | Encoded target 'y' → 0/1 (41188 rows).


Raw shape: (41188, 21)


## 1 — The 9 Engineered Features
These features were hand-crafted based on business reasoning and EDA observations.

In [2]:
df = add_features(df_raw.copy())

print(f"After feature engineering: {df.shape}  (+{df.shape[1] - df_raw.shape[1]} columns)")
print("\nNew features:")
for feat in ENGINEERED_FEATURE_NAMES:
    dtype = str(df[feat].dtype)
    uniq  = df[feat].nunique()
    print(f"  {feat:<40}  dtype={dtype:<12}  unique={uniq}")

2026-05-16 21:20:03 | INFO     | src.features | add_features: added 9 engineered features.


After feature engineering: (41188, 30)  (+9 columns)

New features:
  was_previously_contacted                  dtype=int64         unique=2
  campaign_intensity_group                  dtype=str           unique=3
  age_group                                 dtype=str           unique=3
  economic_stress_index                     dtype=float64       unique=350
  has_any_loan                              dtype=int64         unique=2
  month_order                               dtype=int64         unique=10
  previous_contact_success_flag             dtype=int64         unique=2
  contact_is_cellular                       dtype=int64         unique=2
  client_financial_pressure_flag            dtype=int64         unique=2


In [ ]:
# Visualise each engineered feature vs target
n_cols = 3
n_rows = (len(ENGINEERED_FEATURE_NAMES) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(ENGINEERED_FEATURE_NAMES):
    ax = axes[i]
    rate = df.groupby(feat, observed=True)[TARGET_COL].mean().sort_index()
    n_per_grp = df.groupby(feat, observed=True)[TARGET_COL].count()
    ax.bar(rate.index.astype(str), rate.values * 100, color="#1f77b4")
    ax.axhline(df[TARGET_COL].mean() * 100, color="red", linestyle="--", lw=1.2, label="avg")
    ax.set_title(feat, fontsize=9)
    ax.set_ylabel("Sub. Rate %")
    ax.tick_params(axis="x", rotation=25, labelsize=7)
    ax.legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Subscription Rate by Engineered Feature", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2 — Feature Set A vs Feature Set B
- **Set A** includes `duration` — useful for benchmarking but introduces data leakage  
- **Set B** excludes `duration` — the Realistic Business Model used in production

In [ ]:
lists_a = get_feature_lists(df, TARGET_COL, exclude_duration=False)
lists_b = get_feature_lists(df, TARGET_COL, exclude_duration=True)

print("Feature Set A (with duration):")
print(f"  Numeric  ({len(lists_a['numeric'])}): {lists_a['numeric']}")
print(f"  Categorical ({len(lists_a['categorical'])}): {lists_a['categorical']}")
print(f"  Total: {len(lists_a['numeric']) + len(lists_a['categorical'])}")

print("\nFeature Set B (without duration) — PRODUCTION MODEL:")
print(f"  Numeric  ({len(lists_b['numeric'])}): {lists_b['numeric']}")
print(f"  Categorical ({len(lists_b['categorical'])}): {lists_b['categorical']}")
print(f"  Total: {len(lists_b['numeric']) + len(lists_b['categorical'])}")

## 3 — Save Train / Test Splits to `data/processed/`

In [ ]:
ensure_dir(PROCESSED_DIR)

X_train, X_test, y_train, y_test = split_data(df, TARGET_COL)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}")

# Save
train_df = X_train.copy(); train_df[TARGET_COL] = y_train.values
test_df  = X_test.copy();  test_df[TARGET_COL]  = y_test.values
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv",   index=False)
print(f"\nSaved to {PROCESSED_DIR}/")